## Multilabel Classification of Depression Emotions

### MiniMax-M2.5

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "  #  
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/MiniMax-M2.5_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.siliconflow.cn/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="Pro/MiniMaxAI/MiniMax-M2.5",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 906 条样本


第1轮: 100%|██████████| 906/906 [2:13:02<00:00,  8.81s/it]  

正在按原始 CSV 顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [1]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/MiniMax-M2.5_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.0839
Hamming Loss: 0.2764

Macro Precision: 0.7027
Macro Recall:    0.5902
Macro F1-score:  0.6354

Micro Precision: 0.7330
Micro Recall:    0.6106
Micro F1-score:  0.6662

每个类别的指标：
anger:
  Precision: 0.7452
  Recall:    0.4493
  F1-score:  0.5606
  Support:   345
brain dysfunction (forget):
  Precision: 0.4580
  Recall:    0.3509
  F1-score:  0.3974
  Support:   171
emptiness:
  Precision: 0.6040
  Recall:    0.5279
  F1-score:  0.5634
  Support:   341
hopelessness:
  Precision: 0.8628
  Recall:    0.6151
  F1-score:  0.7182
  Support:   634
loneliness:
  Precision: 0.6878
  Recall:    0.7762
  F1-score:  0.7293
  Support:   420
sadness:
  Precision: 0.8885
  Recall:    0.6514
  F1-score:  0.7517
  Support:   697
suicide intent:
  Precision: 0.7642
  Recall:    0.7543
  F1-score:  0.7592
  Support:   232
worthlessness:
  Precision: 0.6108
  Recall:    0.5968
  F1-score:  0.6037
  Support:   434

总样本数: 906


### MiniMax-M2.7

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "  # 请替换为有效的 API Key
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/MiniMax-M2.7_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="MiniMax-M2.7",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=4000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(15)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

正在按原始 CSV 顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [2]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/MiniMax-M2.7_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.0894
Hamming Loss: 0.2863

Macro Precision: 0.6862
Macro Recall:    0.5921
Macro F1-score:  0.6299

Micro Precision: 0.7167
Micro Recall:    0.6057
Micro F1-score:  0.6565

每个类别的指标：
anger:
  Precision: 0.7059
  Recall:    0.5217
  F1-score:  0.6000
  Support:   345
brain dysfunction (forget):
  Precision: 0.4628
  Recall:    0.3275
  F1-score:  0.3836
  Support:   171
emptiness:
  Precision: 0.5464
  Recall:    0.5865
  F1-score:  0.5658
  Support:   341
hopelessness:
  Precision: 0.8539
  Recall:    0.6451
  F1-score:  0.7350
  Support:   634
loneliness:
  Precision: 0.7058
  Recall:    0.7881
  F1-score:  0.7447
  Support:   420
sadness:
  Precision: 0.8556
  Recall:    0.5782
  F1-score:  0.6901
  Support:   697
suicide intent:
  Precision: 0.7427
  Recall:    0.7716
  F1-score:  0.7569
  Support:   232
worthlessness:
  Precision: 0.6164
  Recall:    0.5184
  F1-score:  0.5632
  Support:   434

总样本数: 906


### Pro/zai-org/GLM-5

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "  # 请替换为有效的 API Key
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/GLM-5_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.siliconflow.cn/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="Pro/zai-org/GLM-5",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 1 条样本


第1轮: 100%|██████████| 1/1 [10:59<00:00, 659.32s/it]

正在按原始 CSV 顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [3]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/GLM-5_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.0773
Hamming Loss: 0.2875

Macro Precision: 0.7146
Macro Recall:    0.5471
Macro F1-score:  0.6071

Micro Precision: 0.7423
Micro Recall:    0.5568
Micro F1-score:  0.6363

每个类别的指标：
anger:
  Precision: 0.7451
  Recall:    0.5507
  F1-score:  0.6333
  Support:   345
brain dysfunction (forget):
  Precision: 0.5357
  Recall:    0.2632
  F1-score:  0.3529
  Support:   171
emptiness:
  Precision: 0.6233
  Recall:    0.3930
  F1-score:  0.4820
  Support:   341
hopelessness:
  Precision: 0.8581
  Recall:    0.6199
  F1-score:  0.7198
  Support:   634
loneliness:
  Precision: 0.6733
  Recall:    0.8095
  F1-score:  0.7351
  Support:   420
sadness:
  Precision: 0.9024
  Recall:    0.4778
  F1-score:  0.6248
  Support:   697
suicide intent:
  Precision: 0.7510
  Recall:    0.7931
  F1-score:  0.7715
  Support:   232
worthlessness:
  Precision: 0.6277
  Recall:    0.4700
  F1-score:  0.5375
  Support:   434

总样本数: 906


### stepfun-ai/Step-3.5-Flash

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "  # 请替换为有效的 API Key
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/Step-3.5-Flash_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.siliconflow.cn/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="stepfun-ai/Step-3.5-Flash",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=8000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 6 条样本


第1轮: 100%|██████████| 6/6 [03:11<00:00, 31.93s/it]

正在按原始 CSV 顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [2]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/Step-3.5-Flash_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.1126
Hamming Loss: 0.2697

Macro Precision: 0.7079
Macro Recall:    0.5648
Macro F1-score:  0.6221

Micro Precision: 0.7536
Micro Recall:    0.5987
Micro F1-score:  0.6672

每个类别的指标：
anger:
  Precision: 0.7246
  Recall:    0.4957
  F1-score:  0.5886
  Support:   345
brain dysfunction (forget):
  Precision: 0.4353
  Recall:    0.2164
  F1-score:  0.2891
  Support:   171
emptiness:
  Precision: 0.6344
  Recall:    0.4223
  F1-score:  0.5070
  Support:   341
hopelessness:
  Precision: 0.8309
  Recall:    0.7129
  F1-score:  0.7674
  Support:   634
loneliness:
  Precision: 0.7273
  Recall:    0.7619
  F1-score:  0.7442
  Support:   420
sadness:
  Precision: 0.8866
  Recall:    0.6169
  F1-score:  0.7276
  Support:   697
suicide intent:
  Precision: 0.7876
  Recall:    0.7672
  F1-score:  0.7773
  Support:   232
worthlessness:
  Precision: 0.6369
  Recall:    0.5253
  F1-score:  0.5758
  Support:   434

总样本数: 906


### baidu/ERNIE-4.5-300B-A47B

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "  # 请替换为有效的 API Key
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/ERNIE-4.5-300B-A47B_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api2.aigcbest.top/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="baidu/ernie-4.5-300b-a47b",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 906 条样本


第1轮:   0%|          | 0/906 [00:00<?, ?it/s]

第1轮: 100%|██████████| 906/906 [40:33<00:00,  2.69s/it]

正在按原始 CSV 顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [5]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/ERNIE-4.5-300B-A47B_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.1137
Hamming Loss: 0.2928

Macro Precision: 0.7591
Macro Recall:    0.4772
Macro F1-score:  0.5766

Micro Precision: 0.7948
Micro Recall:    0.4743
Micro F1-score:  0.5941

每个类别的指标：
anger:
  Precision: 0.7972
  Recall:    0.5014
  F1-score:  0.6157
  Support:   345
brain dysfunction (forget):
  Precision: 0.5417
  Recall:    0.2281
  F1-score:  0.3210
  Support:   171
emptiness:
  Precision: 0.6301
  Recall:    0.4047
  F1-score:  0.4929
  Support:   341
hopelessness:
  Precision: 0.8929
  Recall:    0.4338
  F1-score:  0.5839
  Support:   634
loneliness:
  Precision: 0.8507
  Recall:    0.6786
  F1-score:  0.7550
  Support:   420
sadness:
  Precision: 0.8902
  Recall:    0.4419
  F1-score:  0.5906
  Support:   697
suicide intent:
  Precision: 0.8018
  Recall:    0.7672
  F1-score:  0.7841
  Support:   232
worthlessness:
  Precision: 0.6681
  Recall:    0.3618
  F1-score:  0.4694
  Support:   434

总样本数: 906


### deepseek-ai/DeepSeek-V3.2

In [ ]:
import pandas as pd
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI

# ========== 配置 ==========
key = " "  # 请替换为有效的 API Key
input_json_path = "/data/zhengtianlong/Private/Depressed/DepressionEmo-main/Dataset/test.json"
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/DeepSeek-V3.2_Depression_EIGHT.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# ========== 读取数据 ==========
data = pd.read_json(input_json_path, lines=True)  # JSON lines 格式
# 确保 id 列为字符串类型
data['id'] = data['id'].astype(str)

# ========== 定义情绪分类函数 ==========
def identify_emotions(text):
    """调用大模型，返回帖子中包含的情绪代码（如 '丙, 丁'）"""
    client = OpenAI(
        base_url="https://api.siliconflow.cn/v1",
        api_key=key
    )
    prompt = f"""You are an expert in mental health analysis. Your task is to analyze a social media post and identify which of the following depression-related emotions are expressed in it. Each emotion is represented by a Chinese character code (甲, 乙, 丙, 丁, 戊, 己, 庚, 辛) with corresponding descriptions:

甲: Anger – an intensified emotional response that can be directed inward or outward, contributing to despair and self-criticism.  
乙: Cognitive dysfunction – impairment in normal cognitive functioning, causing forgetfulness, slow thinking, difficulty expressing thoughts.  
丙: Emptiness – a deep emotional void, numbness, lack of vitality, feeling disconnected and an empty future.  
丁: Hopelessness – a susceptibility factor in depressive symptoms, intrinsic to depression.  
戊: Loneliness – profound isolation even in the presence of others, predisposing to depression.  
己: Sadness – a natural emotion triggered by events or losses, a fundamental symptom of depression.  
庚: Suicide intent – conscious desire to end one's life, indicating a serious mental health concern, often overwhelming emotional pain.  
辛: Worthlessness – a profound sense of having little or no value, feeling inadequate or undeserving.

Instructions:
1. Read the given post text carefully.
2. Based on the content, identify all emotions from the list above that are present or implied in the post. Each post is guaranteed to contain at least one of these emotions.
3. Return your answer as a comma-separated list of the corresponding Chinese codes (e.g., "丙, 丁" for emptiness and hopelessness). Do not include any additional text or explanation.
4. Only output the list.

Post: {text}

Emotions expressed:"""

    response = client.chat.completions.create(
        model="deepseek-ai/DeepSeek-V3.2",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# ========== 初始化输出文件 ==========
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as f:
            existing_data = json.load(f)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件损坏或为空，重新初始化
        with open(output_json_path, "w") as f:
            json.dump([], f)
else:
    with open(output_json_path, "w") as f:
        json.dump([], f)

# ========== 重试处理 ==========
max_retries = 5
current_retry = 0
all_ids = set(data['id'].tolist())

while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    remaining_data = data[~data['id'].isin(success_ids)]

    if remaining_data.empty:
        break

    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = row['id']
            text = row['text']
            title = row.get('title', '')
            post = row.get('post', '')
            target = row['emotions']      # 原始情绪列表，如 ['hopelessness', 'sadness']
            label = row['label_id']       # 标签二进制字符串，如 "10100"

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查
            if Id in success_ids:
                continue

            # 调用模型预测
            predict = identify_emotions(text)

            # 构造结果对象
            result = {
                "Id": Id,
                "text": text,
                "title": title,
                "post": post,
                "predict": predict,
                "label": label,
                "target": target   # 保留原始情绪列表用于后续评估
            }

            # 实时写入文件
            with open(output_json_path, "r+") as f:
                file_data = json.load(f)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    f.seek(0)
                    json.dump(file_data, f, ensure_ascii=False, indent=4)
                    f.truncate()
                    success_ids.add(Id)
                else:
                    # 同步内存集合
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 避免请求过快

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            continue

# ========== 按原始顺序排序 ==========
print("正在按原始 CSV 顺序重新排序输出文件...")
id_to_order = {row['id']: idx for idx, row in data.iterrows()}

with open(output_json_path, "r") as f:
    final_data = json.load(f)

final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# ========== 最终统计 ==========
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 906 条样本


第1轮: 100%|██████████| 906/906 [53:08<00:00,  3.52s/it]  

正在按原始 CSV 顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [7]:
import json
import numpy as np
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score, 
                             precision_score, recall_score, 
                             precision_recall_fscore_support)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/DeepSeek-V3.2_Depression_EIGHT.json"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 读取数据 ==========
with open(input_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实情绪列表
    true_emotions = set(item['target'])  # 示例: ["hopelessness", "sadness"]
    # 转换为二进制向量
    true_vec = [1 if name in true_emotions else 0 for name in emotion_names]
    # print(true_vec)
    y_true.append(true_vec)

    # 预测字符串
    pred_str = item.get('predict', '')
    # 提取出现的情绪代码
    pred_codes = {code for code in emotion_codes if code in pred_str}
    # print(pred_codes)
    # 转换为英文名称（用于统一表示，也可直接用代码构建向量）
    pred_emotions = {code_to_name[code] for code in pred_codes}
    # print(pred_emotions)
    pred_vec = [1 if name in pred_emotions else 0 for name in emotion_names]
    # print(pred_vec)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0, labels=range(8)
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")

Exact Match Ratio (准确率): 0.0784
Hamming Loss: 0.2772

Macro Precision: 0.7059
Macro Recall:    0.5760
Macro F1-score:  0.6107

Micro Precision: 0.7263
Micro Recall:    0.6200
Micro F1-score:  0.6690

每个类别的指标：
anger:
  Precision: 0.7791
  Recall:    0.3884
  F1-score:  0.5184
  Support:   345
brain dysfunction (forget):
  Precision: 0.5833
  Recall:    0.2047
  F1-score:  0.3030
  Support:   171
emptiness:
  Precision: 0.5726
  Recall:    0.3930
  F1-score:  0.4661
  Support:   341
hopelessness:
  Precision: 0.8763
  Recall:    0.5473
  F1-score:  0.6738
  Support:   634
loneliness:
  Precision: 0.6065
  Recall:    0.8405
  F1-score:  0.7046
  Support:   420
sadness:
  Precision: 0.8011
  Recall:    0.8608
  F1-score:  0.8299
  Support:   697
suicide intent:
  Precision: 0.7490
  Recall:    0.8362
  F1-score:  0.7902
  Support:   232
worthlessness:
  Precision: 0.6793
  Recall:    0.5369
  F1-score:  0.5997
  Support:   434

总样本数: 906


### Native Qwen3.5-0.8B-Base

In [1]:
import json
import re
import numpy as np
from sklearn.metrics import (
    accuracy_score, hamming_loss, f1_score,
    precision_score, recall_score,
    precision_recall_fscore_support
)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/LlamaFactory/saves/Qwen3.5-0.8B-Base/lora/Native_eval_26-03-18-11-36-32/generated_predictions.jsonl"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 工具函数 ==========
def extract_codes(text, emotion_codes):
    """
    从字符串中提取情绪代码，返回集合。
    例如：
    '<think>...</think>\\n\\n丁, 己' -> {'丁', '己'}
    """
    if text is None:
        return set()
    text = str(text)

    # 去掉 <think>...</think>，避免干扰
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)

    # 只保留出现过的合法代码
    codes = {code for code in emotion_codes if code in text}
    return codes


def codes_to_multihot(code_set, emotion_codes):
    """
    将代码集合转为多标签二进制向量
    """
    return [1 if code in code_set else 0 for code in emotion_codes]


# ========== 读取数据 ==========
data = []
with open(input_json_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data.append(json.loads(line))

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实标签：来自 label 字段
    label_str = item.get("label", "")
    true_codes = extract_codes(label_str, emotion_codes)
    true_vec = codes_to_multihot(true_codes, emotion_codes)
    y_true.append(true_vec)

    # 预测标签：来自 predict 字段
    pred_str = item.get("predict", "")
    pred_codes = extract_codes(pred_str, emotion_codes)
    pred_vec = codes_to_multihot(pred_codes, emotion_codes)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")


Exact Match Ratio (准确率): 0.0121
Hamming Loss: 0.4149

Macro Precision: 0.4848
Macro Recall:    0.5059
Macro F1-score:  0.4742

Micro Precision: 0.5391
Micro Recall:    0.5626
Micro F1-score:  0.5506

每个类别的指标：
anger:
  Precision: 0.3902
  Recall:    0.1855
  F1-score:  0.2515
  Support:   345
brain dysfunction (forget):
  Precision: 0.2500
  Recall:    0.2865
  F1-score:  0.2670
  Support:   171
emptiness:
  Precision: 0.3786
  Recall:    0.8504
  F1-score:  0.5239
  Support:   341
hopelessness:
  Precision: 0.7169
  Recall:    0.8186
  F1-score:  0.7644
  Support:   634
loneliness:
  Precision: 0.4787
  Recall:    0.3214
  F1-score:  0.3846
  Support:   420
sadness:
  Precision: 0.7911
  Recall:    0.7174
  F1-score:  0.7524
  Support:   697
suicide intent:
  Precision: 0.3211
  Recall:    0.4526
  F1-score:  0.3757
  Support:   232
worthlessness:
  Precision: 0.5521
  Recall:    0.4147
  F1-score:  0.4737
  Support:   434

总样本数: 906


### Native Qwen3.5-2B-Base

In [2]:
import json
import re
import numpy as np
from sklearn.metrics import (
    accuracy_score, hamming_loss, f1_score,
    precision_score, recall_score,
    precision_recall_fscore_support
)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/LlamaFactory/saves/Qwen3.5-2B-Base/lora/Native_eval_26-03-18-11-36-32/generated_predictions.jsonl"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 工具函数 ==========
def extract_codes(text, emotion_codes):
    """
    从字符串中提取情绪代码，返回集合。
    例如：
    '<think>...</think>\\n\\n丁, 己' -> {'丁', '己'}
    """
    if text is None:
        return set()
    text = str(text)

    # 去掉 <think>...</think>，避免干扰
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)

    # 只保留出现过的合法代码
    codes = {code for code in emotion_codes if code in text}
    return codes


def codes_to_multihot(code_set, emotion_codes):
    """
    将代码集合转为多标签二进制向量
    """
    return [1 if code in code_set else 0 for code in emotion_codes]


# ========== 读取数据 ==========
data = []
with open(input_json_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data.append(json.loads(line))

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实标签：来自 label 字段
    label_str = item.get("label", "")
    true_codes = extract_codes(label_str, emotion_codes)
    true_vec = codes_to_multihot(true_codes, emotion_codes)
    y_true.append(true_vec)

    # 预测标签：来自 predict 字段
    pred_str = item.get("predict", "")
    pred_codes = extract_codes(pred_str, emotion_codes)
    pred_vec = codes_to_multihot(pred_codes, emotion_codes)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")


Exact Match Ratio (准确率): 0.0066
Hamming Loss: 0.4687

Macro Precision: 0.4772
Macro Recall:    0.4544
Macro F1-score:  0.4343

Micro Precision: 0.4800
Micro Recall:    0.4511
Micro F1-score:  0.4651

每个类别的指标：
anger:
  Precision: 0.3850
  Recall:    0.2087
  F1-score:  0.2707
  Support:   345
brain dysfunction (forget):
  Precision: 0.2176
  Recall:    0.3333
  F1-score:  0.2633
  Support:   171
emptiness:
  Precision: 0.3771
  Recall:    0.7918
  F1-score:  0.5109
  Support:   341
hopelessness:
  Precision: 0.7026
  Recall:    0.4732
  F1-score:  0.5655
  Support:   634
loneliness:
  Precision: 0.5166
  Recall:    0.3714
  F1-score:  0.4321
  Support:   420
sadness:
  Precision: 0.7926
  Recall:    0.4275
  F1-score:  0.5555
  Support:   697
suicide intent:
  Precision: 0.3133
  Recall:    0.6078
  F1-score:  0.4135
  Support:   232
worthlessness:
  Precision: 0.5126
  Recall:    0.4217
  F1-score:  0.4627
  Support:   434

总样本数: 906


### Native Qwen3.5-4B-Base

In [3]:
import json
import re
import numpy as np
from sklearn.metrics import (
    accuracy_score, hamming_loss, f1_score,
    precision_score, recall_score,
    precision_recall_fscore_support
)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/LlamaFactory/saves/Qwen3.5-4B-Base/lora/Native_eval_26-03-18-11-36-32/generated_predictions.jsonl"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 工具函数 ==========
def extract_codes(text, emotion_codes):
    """
    从字符串中提取情绪代码，返回集合。
    例如：
    '<think>...</think>\\n\\n丁, 己' -> {'丁', '己'}
    """
    if text is None:
        return set()
    text = str(text)

    # 去掉 <think>...</think>，避免干扰
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)

    # 只保留出现过的合法代码
    codes = {code for code in emotion_codes if code in text}
    return codes


def codes_to_multihot(code_set, emotion_codes):
    """
    将代码集合转为多标签二进制向量
    """
    return [1 if code in code_set else 0 for code in emotion_codes]


# ========== 读取数据 ==========
data = []
with open(input_json_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data.append(json.loads(line))

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实标签：来自 label 字段
    label_str = item.get("label", "")
    true_codes = extract_codes(label_str, emotion_codes)
    true_vec = codes_to_multihot(true_codes, emotion_codes)
    y_true.append(true_vec)

    # 预测标签：来自 predict 字段
    pred_str = item.get("predict", "")
    pred_codes = extract_codes(pred_str, emotion_codes)
    pred_vec = codes_to_multihot(pred_codes, emotion_codes)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")


Exact Match Ratio (准确率): 0.0320
Hamming Loss: 0.4266

Macro Precision: 0.5011
Macro Recall:    0.7152
Macro F1-score:  0.5726

Micro Precision: 0.5196
Micro Recall:    0.7358
Micro F1-score:  0.6091

每个类别的指标：
anger:
  Precision: 0.4296
  Recall:    0.4957
  F1-score:  0.4603
  Support:   345
brain dysfunction (forget):
  Precision: 0.1981
  Recall:    0.4971
  F1-score:  0.2833
  Support:   171
emptiness:
  Precision: 0.3855
  Recall:    0.9384
  F1-score:  0.5465
  Support:   341
hopelessness:
  Precision: 0.7055
  Recall:    0.8312
  F1-score:  0.7632
  Support:   634
loneliness:
  Precision: 0.5536
  Recall:    0.7500
  F1-score:  0.6370
  Support:   420
sadness:
  Precision: 0.8171
  Recall:    0.7116
  F1-score:  0.7607
  Support:   697
suicide intent:
  Precision: 0.3853
  Recall:    0.7672
  F1-score:  0.5130
  Support:   232
worthlessness:
  Precision: 0.5337
  Recall:    0.7304
  F1-score:  0.6167
  Support:   434

总样本数: 906


### Native Qwen3.5-9B-Base

In [4]:
import json
import re
import numpy as np
from sklearn.metrics import (
    accuracy_score, hamming_loss, f1_score,
    precision_score, recall_score,
    precision_recall_fscore_support
)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/LlamaFactory/saves/Qwen3.5-9B-Base/lora/Native_eval_26-03-18-11-36-32/generated_predictions.jsonl"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 工具函数 ==========
def extract_codes(text, emotion_codes):
    """
    从字符串中提取情绪代码，返回集合。
    例如：
    '<think>...</think>\\n\\n丁, 己' -> {'丁', '己'}
    """
    if text is None:
        return set()
    text = str(text)

    # 去掉 <think>...</think>，避免干扰
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)

    # 只保留出现过的合法代码
    codes = {code for code in emotion_codes if code in text}
    return codes


def codes_to_multihot(code_set, emotion_codes):
    """
    将代码集合转为多标签二进制向量
    """
    return [1 if code in code_set else 0 for code in emotion_codes]


# ========== 读取数据 ==========
data = []
with open(input_json_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data.append(json.loads(line))

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实标签：来自 label 字段
    label_str = item.get("label", "")
    true_codes = extract_codes(label_str, emotion_codes)
    true_vec = codes_to_multihot(true_codes, emotion_codes)
    y_true.append(true_vec)

    # 预测标签：来自 predict 字段
    pred_str = item.get("predict", "")
    pred_codes = extract_codes(pred_str, emotion_codes)
    pred_vec = codes_to_multihot(pred_codes, emotion_codes)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")


Exact Match Ratio (准确率): 0.0320
Hamming Loss: 0.4269

Macro Precision: 0.5085
Macro Recall:    0.7328
Macro F1-score:  0.5824

Micro Precision: 0.5193
Micro Recall:    0.7407
Micro F1-score:  0.6105

每个类别的指标：
anger:
  Precision: 0.4552
  Recall:    0.5884
  F1-score:  0.5133
  Support:   345
brain dysfunction (forget):
  Precision: 0.1944
  Recall:    0.6491
  F1-score:  0.2992
  Support:   171
emptiness:
  Precision: 0.4065
  Recall:    0.8035
  F1-score:  0.5399
  Support:   341
hopelessness:
  Precision: 0.7477
  Recall:    0.7571
  F1-score:  0.7524
  Support:   634
loneliness:
  Precision: 0.5441
  Recall:    0.7929
  F1-score:  0.6453
  Support:   420
sadness:
  Precision: 0.8105
  Recall:    0.7547
  F1-score:  0.7816
  Support:   697
suicide intent:
  Precision: 0.3890
  Recall:    0.7931
  F1-score:  0.5220
  Support:   232
worthlessness:
  Precision: 0.5207
  Recall:    0.7235
  F1-score:  0.6056
  Support:   434

总样本数: 906


### Native Qwen3.5-35B-A3B-Base

In [1]:
import json
import re
import numpy as np
from sklearn.metrics import (
    accuracy_score, hamming_loss, f1_score,
    precision_score, recall_score,
    precision_recall_fscore_support
)

# ========== 配置 ==========
input_json_path = "/data/zhengtianlong/Private/Depressed/LlamaFactory/saves/Qwen3.5-35B-A3B-Base/lora/Native_eval_26-03-18-11-36-32/generated_predictions.jsonl"

# 情绪代码到英文名称的映射（顺序固定，用于二进制向量）
emotion_codes = ['甲', '乙', '丙', '丁', '戊', '己', '庚', '辛']
emotion_names = [
    'anger',
    'brain dysfunction (forget)',
    'emptiness',
    'hopelessness',
    'loneliness',
    'sadness',
    'suicide intent',
    'worthlessness'
]
code_to_name = dict(zip(emotion_codes, emotion_names))

# ========== 工具函数 ==========
def extract_codes(text, emotion_codes):
    """
    从字符串中提取情绪代码，返回集合。
    例如：
    '<think>...</think>\\n\\n丁, 己' -> {'丁', '己'}
    """
    if text is None:
        return set()
    text = str(text)

    # 去掉 <think>...</think>，避免干扰
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)

    # 只保留出现过的合法代码
    codes = {code for code in emotion_codes if code in text}
    return codes


def codes_to_multihot(code_set, emotion_codes):
    """
    将代码集合转为多标签二进制向量
    """
    return [1 if code in code_set else 0 for code in emotion_codes]


# ========== 读取数据 ==========
data = []
with open(input_json_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data.append(json.loads(line))

# ========== 解析预测和真实标签 ==========
y_true = []   # 真实二进制向量
y_pred = []   # 预测二进制向量

for item in data:
    # 真实标签：来自 label 字段
    label_str = item.get("label", "")
    true_codes = extract_codes(label_str, emotion_codes)
    true_vec = codes_to_multihot(true_codes, emotion_codes)
    y_true.append(true_vec)

    # 预测标签：来自 predict 字段
    pred_str = item.get("predict", "")
    pred_codes = extract_codes(pred_str, emotion_codes)
    pred_vec = codes_to_multihot(pred_codes, emotion_codes)
    y_pred.append(pred_vec)

# 转换为 numpy 数组
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# ========== 计算指标 ==========
# 1. Exact Match Ratio (准确率)
exact_match = accuracy_score(y_true, y_pred)
print(f"Exact Match Ratio (准确率): {exact_match:.4f}")

# 2. Hamming Loss
h_loss = hamming_loss(y_true, y_pred)
print(f"Hamming Loss: {h_loss:.4f}")

# 3. 宏平均指标
precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
print(f"\nMacro Precision: {precision_macro:.4f}")
print(f"Macro Recall:    {recall_macro:.4f}")
print(f"Macro F1-score:  {f1_macro:.4f}")

# 4. 微平均指标
precision_micro = precision_score(y_true, y_pred, average='micro', zero_division=0)
recall_micro = recall_score(y_true, y_pred, average='micro', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
print(f"\nMicro Precision: {precision_micro:.4f}")
print(f"Micro Recall:    {recall_micro:.4f}")
print(f"Micro F1-score:  {f1_micro:.4f}")

# 5. 每个类别的精确率、召回率、F1-score
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, zero_division=0
)

print("\n每个类别的指标：")
for i, name in enumerate(emotion_names):
    print(f"{name}:")
    print(f"  Precision: {precision[i]:.4f}")
    print(f"  Recall:    {recall[i]:.4f}")
    print(f"  F1-score:  {f1[i]:.4f}")
    print(f"  Support:   {support[i]}")

# 可选：输出总体统计
print(f"\n总样本数: {len(data)}")


Exact Match Ratio (准确率): 0.0541
Hamming Loss: 0.3764

Macro Precision: 0.5361
Macro Recall:    0.6798
Macro F1-score:  0.5855

Micro Precision: 0.5670
Micro Recall:    0.7056
Micro F1-score:  0.6287

每个类别的指标：
anger:
  Precision: 0.5471
  Recall:    0.5217
  F1-score:  0.5341
  Support:   345
brain dysfunction (forget):
  Precision: 0.2270
  Recall:    0.3743
  F1-score:  0.2826
  Support:   171
emptiness:
  Precision: 0.4309
  Recall:    0.6950
  F1-score:  0.5320
  Support:   341
hopelessness:
  Precision: 0.7263
  Recall:    0.8707
  F1-score:  0.7920
  Support:   634
loneliness:
  Precision: 0.5308
  Recall:    0.8619
  F1-score:  0.6570
  Support:   420
sadness:
  Precision: 0.8067
  Recall:    0.6585
  F1-score:  0.7251
  Support:   697
suicide intent:
  Precision: 0.4253
  Recall:    0.8707
  F1-score:  0.5714
  Support:   232
worthlessness:
  Precision: 0.5948
  Recall:    0.5853
  F1-score:  0.5900
  Support:   434

总样本数: 906
